# HELM — Dataset Exploration (UCM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimchev/HELM/blob/master/notebooks/eda.ipynb)

Exploratory data analysis of the UC Merced Land Use dataset: images, label distributions, hierarchy structure, and multi-label co-occurrence.

## 1. Setup

In [ ]:
%cd /content
!rm -rf HELM
!git clone https://github.com/marjanstoimchev/HELM.git
%cd HELM
!pip install -q -r requirements.txt

import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html 2>/dev/null || echo "PyG extensions installed via fallback"
print("Setup complete!")

In [ ]:
import os, sys, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from collections import Counter

sys.path.insert(0, ".")
from data.dataset_pipeline import DatasetPipeline
from data.hierarchy import process_hierarchy_config, extract_paths, extend_ys, build_hierarchy_mapping

plt.rcParams.update({
    "font.family": "serif", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 120, "axes.titleweight": "bold",
})

## 2. Load dataset

In [ ]:
DATASET = "ucm"
DATASET_CONFIG = f"configs/dataset/{DATASET}.yaml"

with open(DATASET_CONFIG) as f:
    ds_cfg = yaml.safe_load(f)

pipeline = DatasetPipeline(yaml_path=DATASET_CONFIG, seed=42, cache_dir="./Datasets/mlc_datasets_npy")
outputs = pipeline.run_pipeline(fraction_labeled=None)

images = np.concatenate([outputs['X'][0], outputs['X_val'], outputs['X_te']], axis=0)
labels = np.concatenate([outputs['X'][1], outputs['Y_val'], outputs['Y_te']], axis=0)

leaf_labels = ds_cfg["leaf_labels"]
print(f"Dataset: {ds_cfg['name']}")
print(f"Total images: {len(images)}")
print(f"Image shape: {images[0].shape}")
print(f"Labels: {len(leaf_labels)} classes")

## 3. Hierarchy tree

In [ ]:
def print_tree(node, prefix="", is_last=True):
    keys = list(node.keys())
    for i, key in enumerate(keys):
        last = (i == len(keys) - 1)
        connector = "\u2514\u2500\u2500 " if last else "\u251c\u2500\u2500 "
        is_leaf = not node[key]
        suffix = " \u25cf" if is_leaf else ""
        print(f"{prefix}{connector}{key}{suffix}")
        if node[key]:
            extension = "    " if last else "\u2502   "
            print_tree(node[key], prefix + extension, last)

print(f"UCM Label Hierarchy ({ds_cfg['num_classes']} leaf classes, \u25cf = leaf):\n")
print_tree(ds_cfg["hierarchy"])

## 4. Sample images

Random images from the dataset with their multi-label annotations.

In [ ]:
N_SHOW = 8
indices = np.random.choice(len(images), N_SHOW, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, idx in enumerate(indices):
    ax = axes[i // 4][i % 4]
    img = images[idx]
    if img.max() > 1:
        img = img.astype(np.float32) / 255.0
    img = np.moveaxis(img, 0, -1)
    ax.imshow(np.clip(img, 0, 1))
    ax.set_xticks([]); ax.set_yticks([])
    active = [leaf_labels[j] for j in np.where(labels[idx] > 0)[0]]
    ax.set_title(", ".join(active), fontsize=9, wrap=True)

fig.suptitle(f"UCM \u2014 Sample Images", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 5. Label distribution

Frequency of each label across the entire dataset.

In [ ]:
label_counts = labels.sum(axis=0).astype(int)

fig, ax = plt.subplots(figsize=(12, 5))
sorted_idx = np.argsort(label_counts)[::-1]
sorted_names = [leaf_labels[i] for i in sorted_idx]
sorted_counts = label_counts[sorted_idx]

bars = ax.bar(range(len(sorted_names)), sorted_counts, color="#2E86AB", edgecolor="white")
ax.set_xticks(range(len(sorted_names)))
ax.set_xticklabels(sorted_names, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Count")
ax.set_title(f"Label Frequency \u2014 {ds_cfg['name']} ({len(images)} images)")

# Add count on top of bars
for bar, count in zip(bars, sorted_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

## 6. Labels per image

Distribution of how many labels each image has.

In [ ]:
labels_per_image = labels.sum(axis=1).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
ax = axes[0]
counts = Counter(labels_per_image)
x = sorted(counts.keys())
y = [counts[k] for k in x]
ax.bar(x, y, color="#E8871E", edgecolor="white")
ax.set_xlabel("Number of labels per image")
ax.set_ylabel("Count")
ax.set_title("Labels per Image")

# Stats
ax = axes[1]
stats_text = (
    f"Min labels:    {labels_per_image.min()}\n"
    f"Max labels:    {labels_per_image.max()}\n"
    f"Mean labels:   {labels_per_image.mean():.2f}\n"
    f"Median labels: {np.median(labels_per_image):.1f}\n"
    f"Total images:  {len(labels_per_image)}\n"
    f"Total labels:  {len(leaf_labels)}"
)
ax.text(0.3, 0.5, stats_text, transform=ax.transAxes, fontsize=13,
        verticalalignment="center", fontfamily="monospace",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f0f0", edgecolor="#ccc"))
ax.set_axis_off()
ax.set_title("Summary Statistics")

fig.suptitle(f"UCM \u2014 Multi-label Statistics", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 7. Label co-occurrence matrix

Shows how often pairs of labels appear together in the same image.

In [ ]:
co_occurrence = labels.T @ labels
np.fill_diagonal(co_occurrence, 0)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(co_occurrence, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(leaf_labels)))
ax.set_yticks(range(len(leaf_labels)))
ax.set_xticklabels(leaf_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(leaf_labels, fontsize=8)
plt.colorbar(im, ax=ax, label="Co-occurrence count")
ax.set_title(f"Label Co-occurrence \u2014 UCM", fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Hierarchy-extended labels

In HMLC mode, each active leaf label also activates its ancestor nodes. This shows the extended label distribution.

In [ ]:
h_out = process_hierarchy_config(DATASET_CONFIG)
leaf_paths = h_out.leaf_paths.to_dict()
intermediate_paths = h_out.intermediate_paths.to_dict()
ordered_lp = {k: leaf_paths[k] for k in leaf_labels}
final_strings = {**ordered_lp, **intermediate_paths}
all_names = list(final_strings.keys())

labels_h = np.concatenate([outputs['X'][2], outputs['Y_val_h'], outputs['Y_te_h']], axis=0)

fig, ax = plt.subplots(figsize=(14, 6))
h_counts = labels_h.sum(axis=0).astype(int)
colors = ["#2E86AB" if name in leaf_labels else "#E8871E" for name in all_names]

bars = ax.bar(range(len(all_names)), h_counts, color=colors, edgecolor="white")
ax.set_xticks(range(len(all_names)))
ax.set_xticklabels(all_names, rotation=60, ha="right", fontsize=7)
ax.set_ylabel("Count")
ax.set_title(f"Extended Label Frequency \u2014 {len(all_names)} nodes ({h_out.num_classes} leaf + {len(intermediate_paths)} ancestors)")
ax.legend(
    [Patch(color="#2E86AB"), Patch(color="#E8871E")],
    ["Leaf label", "Ancestor node"],
    loc="upper right",
)

plt.tight_layout()
plt.show()